In [10]:
!pip install openai datasets pandas python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.1 MB/s eta 0:00:00a 0:00:01
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.9/315.9 kB 8.9 MB/s eta 0:00:00
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [92]:
import time
import json
import csv
import os
import pandas as pd
from dataclasses import dataclass, field, asdict
from typing import Optional
from openai import OpenAI  
from dotenv import load_dotenv
load_dotenv()

True

In [121]:
LLM_CONFIG = {
    "model": "gpt-4o-mini",
    "max_tokens": 1024,
    "temperature": 0.1,
}

COST_PER_1K_TOKENS = {
    "gpt-4o-mini":      {"input": 0.00015, "output": 0.0006},
    "qwen-2-72b":       {"input": 0.0009,  "output": 0.0009},
    "gemini-1.5-flash": {"input": 0.000075,"output": 0.0003},
    "llama-3.1-70b":    {"input": 0.00059, "output": 0.00079},
}

PATHS = {
    "gaia":       "../datasets/processed/processed_gaia.parquet",
    # "swe_bench":  "../datasets/processed/processed_swe_bench.parquet",
    # "math_hard":  "../datasets/processed/processed_math_hard.parquet",
    # "agentbench": "../datasets/processed/processed_agentbench.parquet",
}

In [122]:
RESULTS_DIR = "../datasets/baseline/"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [123]:
@dataclass
class Query:
    id: str
    dataset: str
    question: str
    ground_truth: str
    metadata: dict = field(default_factory=dict)

@dataclass
class Result:
    query_id: str
    dataset: str
    question: str
    ground_truth: str
    predicted: str
    is_correct: bool
    input_tokens: int
    output_tokens: int
    total_tokens: int
    cost_usd: float
    time_seconds: float
    model: str
    level: Optional[str] = None
    error: Optional[str] = None

In [124]:
def load_gaia(path: str) -> list[Query]:
    df = pd.read_parquet(path)

    print(f"[GAIA] Loaded {len(df)} rows")
    print(f"[GAIA] Columns: {df.columns.tolist()}")   # inspect once, then lock columns below

    queries = []
    for _, row in df.iterrows():
        queries.append(Query(
            id           = str(row.get("id",   row.name)),   # fallback to index
            dataset      = "GAIA",
            question     = str(row["query"]),                   # ← adjust col name if needed
            ground_truth = str(row["answer"]),               # ← adjust col name if needed
            metadata     = {
                "level": row.get("level", None),
                "file":  row.get("file_name", None),
            }
        ))
    return queries

In [125]:
load_gaia(PATHS["gaia"])

[GAIA] Loaded 165 rows
[GAIA] Columns: ['id', 'query', 'answer', 'level', 'annotator_steps', 'annotator_tools', 'file_name', 'steps_num', 'tool_num', 'time_taken']


[Query(id='c61d22de-5f6c-4958-a7f6-5e9707bd3466', dataset='GAIA', question='A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure with three axes, where each axis has a label word at both ends. Which of these words is used to describe a type of society in a Physics and Society article submitted to arXiv.org on August 11, 2016?', ground_truth='egalitarian', metadata={'level': '2', 'file': ''}),
 Query(id='17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc', dataset='GAIA', question='I’m researching species that became invasive after people who kept them as pets released them. There’s a certain species of fish that was popularized as a pet by being the main character of the movie Finding Nemo. According to the USGS, where was this fish found as a nonnative species, before the year 2020? I need the answer formatted as the five-digit zip codes of the places the species was found, separated by commas if there is more than one place.', ground_truth='34689', met

In [126]:
def load_unified_dataset() -> list[Query]:
    queries = []
    queries += load_gaia(PATHS["gaia"])

    # Uncomment as you add more processed datasets:
    # queries += load_swe_bench(PATHS["swe_bench"])
    # queries += load_math_hard(PATHS["math_hard"])
    # queries += load_agentbench(PATHS["agentbench"])

    print(f"\nTotal queries loaded: {len(queries)}")
    return queries

In [127]:
load_unified_dataset()

[GAIA] Loaded 165 rows
[GAIA] Columns: ['id', 'query', 'answer', 'level', 'annotator_steps', 'annotator_tools', 'file_name', 'steps_num', 'tool_num', 'time_taken']

Total queries loaded: 165


[Query(id='c61d22de-5f6c-4958-a7f6-5e9707bd3466', dataset='GAIA', question='A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure with three axes, where each axis has a label word at both ends. Which of these words is used to describe a type of society in a Physics and Society article submitted to arXiv.org on August 11, 2016?', ground_truth='egalitarian', metadata={'level': '2', 'file': ''}),
 Query(id='17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc', dataset='GAIA', question='I’m researching species that became invasive after people who kept them as pets released them. There’s a certain species of fish that was popularized as a pet by being the main character of the movie Finding Nemo. According to the USGS, where was this fish found as a nonnative species, before the year 2020? I need the answer formatted as the five-digit zip codes of the places the species was found, separated by commas if there is more than one place.', ground_truth='34689', met

In [131]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

SYSTEM_PROMPT = """You are an expert question answering system evaluated on the GAIA benchmark.

## Output Rules
- Return ONLY the final answer — nothing else
- Answers are always one of: a single word, a number, a short phrase, or a name
- No explanations, no sentences, no punctuation at the end
- No preamble like "The answer is..." or "Based on..."
- If the answer is a number, return just a single number after one comma dont return (e.g. 42, 3.14)
- If the answer is a name, return just the name (e.g. Paris, Einstein)
- If the answer is a word, return just that word (e.g. egalitarian, blue)
- If the answer is a short phrase, return just that phrase (e.g. "New York", "machine learning")

## Now answer the following question with ONLY the final answer:
"""
def call_direct_llm(query: Query) -> Result:
    model   = LLM_CONFIG["model"]
    start   = time.perf_counter()
    error   = None
    predicted = ""
    usage   = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

    try:
        response = client.chat.completions.create(
            model=model,
            max_tokens=LLM_CONFIG["max_tokens"],
            temperature=LLM_CONFIG["temperature"],
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": query.question},
            ],
        )
        predicted = response.choices[0].message.content.strip()
        usage = {
            "prompt_tokens":     response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens":      response.usage.total_tokens,
        }
    except Exception as e:
        error = str(e)
        error_type = type(e).__name__

    elapsed = time.perf_counter() - start

    rates = COST_PER_1K_TOKENS.get(model, {"input": 0, "output": 0})
    cost  = (
        usage["prompt_tokens"]     / 1000 * rates["input"] +
        usage["completion_tokens"] / 1000 * rates["output"]
    )

    is_correct = predicted.strip().lower() == query.ground_truth.strip().lower()

    return Result(
        query_id     = query.id,
        dataset      = query.dataset,
        question     = query.question,
        ground_truth = query.ground_truth,
        predicted    = predicted,
        is_correct   = is_correct,
        input_tokens = usage["prompt_tokens"],
        output_tokens= usage["completion_tokens"],
        total_tokens = usage["total_tokens"],
        cost_usd     = round(cost, 6),
        time_seconds = round(elapsed, 3),
        model        = model,
        level        = query.metadata.get("level"),
        error        = error,
    )

In [143]:
def run_baseline(queries: list[Query]) -> list[Result]:
    results = []
    for i, q in enumerate(queries):
        print(f"[{i+1}/{len(queries)}] {q.dataset} | {q.id}")
        print(f"   ❓ Question : {q.question[:100]}...")   # trim long questions
        print(f"   ✅ Expected : {q.ground_truth}")

        r = call_direct_llm(q)
        results.append(r)

        status = "✓" if r.is_correct else "✗"
        print(f"   🤖 Predicted: {r.predicted}")
        print(f"   {status} correct={r.is_correct}  tokens={r.total_tokens}  "
              f"cost=${r.cost_usd:.5f}  time={r.time_seconds}s")
        if r.error:
            print(f"   ⚠ error: {r.error} ({r.error_type})")
        print("-" * 60)

    return results

In [144]:
queries = load_unified_dataset()
results = run_baseline(queries)

[GAIA] Loaded 165 rows
[GAIA] Columns: ['id', 'query', 'answer', 'level', 'annotator_steps', 'annotator_tools', 'file_name', 'steps_num', 'tool_num', 'time_taken']

Total queries loaded: 165
[1/165] GAIA | c61d22de-5f6c-4958-a7f6-5e9707bd3466
   ❓ Question : A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure w...
   ✅ Expected : egalitarian
   🤖 Predicted: egalitarian
   ✓ correct=True  tokens=269  cost=$0.00004  time=0.886s
------------------------------------------------------------
[2/165] GAIA | 17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc
   ❓ Question : I’m researching species that became invasive after people who kept them as pets released them. There...
   ✅ Expected : 34689
   🤖 Predicted: 33139, 33140, 33141, 33142, 33143
   ✗ correct=False  tokens=311  cost=$0.00006  time=0.804s
------------------------------------------------------------
[3/165] GAIA | 04a04a9b-226c-43fd-b319-d5e89743676f
   ❓ Question : If we assume all articles publ

In [149]:

def evaluate(results: list[Result]):
    total   = len(results)
    correct = sum(r.is_correct for r in results)

    print("\n─── BASELINE SUMMARY ───────────────────────────")
    print(f"  Model         : {results[0].model}")
    print(f"  Total queries : {total}")
    print(f"  Accuracy      : {correct}/{total} = {correct/total:.1%}")
    print(f"  Total tokens  : {sum(r.total_tokens  for r in results):,}")
    print(f"  Total cost    : ${sum(r.cost_usd     for r in results):.4f}")
    print(f"  Avg latency   : {sum(r.time_seconds  for r in results)/total:.2f}s")
    print(f"  Errors        : {sum(1 for r in results if r.error)}")

    # Per-dataset breakdown
    print("\n─── PER DATASET ─────────────────────────────────")
    for ds in sorted({r.dataset for r in results}):
        ds_res = [r for r in results if r.dataset == ds]
        acc = sum(r.is_correct for r in ds_res) / len(ds_res)
        print(f"  [{ds}]  accuracy={acc:.1%}  n={len(ds_res)}"
              f"  cost=${sum(r.cost_usd for r in ds_res):.4f}")

    # GAIA level breakdown (if metadata present)
    gaia_res = [r for r in results if r.dataset == "GAIA"]
    if gaia_res:
        df_eval = pd.DataFrame([asdict(r) for r in gaia_res])
        if "level" in df_eval.columns:
            print("\n─── GAIA BY LEVEL ───────────────────────────────")
            # print(df_eval.groupby("level")["is_correct"].mean().to_string())
            print(
            (df_eval.groupby("level")["is_correct"].mean() * 100)
            .round(2)
            .astype(str) + "%"
)



In [150]:
evaluate(results)


─── BASELINE SUMMARY ───────────────────────────
  Model         : gpt-4o-mini
  Total queries : 165
  Accuracy      : 9/165 = 5.5%
  Total tokens  : 44,315
  Total cost    : $0.0069
  Avg latency   : 0.74s
  Errors        : 0

─── PER DATASET ─────────────────────────────────
  [GAIA]  accuracy=5.5%  n=165  cost=$0.0069

─── GAIA BY LEVEL ───────────────────────────────
level
1    5.66%
2    6.98%
3     0.0%
Name: is_correct, dtype: str


In [151]:
def save_results(results: list[Result]):
    model_tag = LLM_CONFIG["model"].replace("/", "-")
    
    # Save path: datasets/baseline/gaia/v1_{model_name}.parquet
    save_dir = os.path.join("..", "datasets", "baseline", "gaia")
    os.makedirs(save_dir, exist_ok=True)
    
    parquet_path = os.path.join(save_dir, f"v1_{model_tag}.parquet")
    
    pd.DataFrame([asdict(r) for r in results]).to_parquet(parquet_path, index=False)
    
    print(f"\n✅ Saved → {parquet_path}")
    print(f"   Rows    : {len(results)}")
    print(f"   Model   : {model_tag}")
    print(f"   Correct : {sum(r.is_correct for r in results)}/{len(results)}")

In [152]:
save_results(results)


✅ Saved → ../datasets/baseline/gaia/v1_gpt-4o-mini.parquet
   Rows    : 165
   Model   : gpt-4o-mini
   Correct : 9/165
